# 1. Mount google drive

In [1]:
# Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Set working directory

import os, sys, shutil
repo_root = "/content/drive/Othercomputers/My Laptop/SmartHVAC-Studio"
colab_dir = os.path.join(repo_root, "colab")

# Ensure eplus utility is copied for imports
eplus_src = os.path.join(repo_root, "EnergyPlus utility")
eplus_dest = os.path.join(colab_dir, "eplus")
if not os.path.exists(eplus_dest):
    shutil.copytree(eplus_src, eplus_dest)

os.chdir(colab_dir)
sys.path.append(colab_dir)
print(f"Working directory set to: {os.getcwd()}")


Working directory set to: /content/drive/Othercomputers/My Laptop/SmartHVAC-Studio/colab


## Install openstudio


In [3]:
# Run this in a new cell at the top of your notebook:
from eplus import prepare_colab_eplus
prepare_colab_eplus()


## Setup NGROK

In [4]:
!pip install pyngrok nest-asyncio fastapi uvicorn pydantic


In [10]:
from backend.fastapi_server import start_server
start_server(port=8000)


************************************************************
🚀 SMART HVAC BACKEND IS LIVE!
🔗 COPY THIS URL TO YOUR WEB DASHBOARD: https://glaucoma-concert-outer.ngrok-free.dev
📄 VIEW SYSTEM LOGS AT: https://glaucoma-concert-outer.ngrok-free.dev/results/backend.log
💡 TO VIEW LIVE LOGS IN COLAB RUN: !tail -n 100 -f /tmp/smarthvac_sim_runs/backend.log
************************************************************
Server is running in the background. You can continue using the notebook.


## 2. Ollama local run

In [5]:
# 1. Install missing dependencies & GPU utils
!sudo apt-get install -y zstd
!sudo apt update && sudo apt install pciutils lshw -y

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 88 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (7,106 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 122464 files and directories currently

In [6]:
# 2. Install Ollama Linux Engine
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [7]:
# 3. Install Python SDK
!pip install ollama

In [8]:
# 2. Safely copy the massive brain from Drive to Colab's fast internal disk (~90 seconds)
print("Copying model from Drive to local disk...")
!mkdir -p /root/.ollama/models
!cp -r "/content/drive/Othercomputers/My Laptop/SmartHVAC-Studio/colab/ollama_models/"* /root/.ollama/models/
print("Copy complete!")

Copying model from Drive to local disk...
Copy complete!


In [4]:
# 4. Set host environment variables to allow connections
import os
import time
os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'

In [5]:
# 3. Boot Server normally (it will use the fast internal disk, NOT Drive)
!nohup ollama serve > ollama.log 2>&1 &
time.sleep(5)
print("✅ Ollama Server Awake and ready!")

✅ Ollama Server Awake and ready!


In [6]:
# 6. Test if it is running
!curl http://localhost:11434

Ollama is running

In [7]:
!ollama list

NAME          ID              SIZE      MODIFIED       
gemma3:12b    f4031aab637d    8.1 GB    53 minutes ago    


## Install Ollama models

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
os.environ['OLLAMA_MODELS'] = '/content/drive/Othercomputers/My Laptop/SmartHVAC-Studio/colab/ollama_models/'


In [ ]:
!ollama pull gemma3:12b


In [ ]:
import subprocess
# Remove gemma manifest
!rm -rf "'/content/drive/Othercomputers/My Laptop/SmartHVAC-Studio/colab/ollama_models/manifests/registry.ollama.ai/library/gemma3"


In [ ]:
# Easiest: just run ollama rm while OLLAMA_MODELS points to Drive
import os
os.environ["OLLAMA_MODELS"] = "/content/drive/Othercomputers/My Laptop/SmartHVAC-Studio/colab/ollama_models"
!ollama rm gemma3:4b


deleted 'gemma3:4b'


## Test model

In [8]:
"""### Testing on Text Input"""

import ollama
response = ollama.chat(model='gemma3:12b', messages=[
  {'role': 'user', 'content': 'what LLM model are you, answer short'},
])
print(response['message']['content'])

I'm Gemma, a large language model created by Google DeepMind.


## 2. Setup Environment

In [9]:
# 1. Install Backend Dependencies
!pip install -r requirements.txt

# 2. Install Advisor's EnergyPlus Utility (from GitHub)
# Note: This installs the specific dev branch required for EKF hooks
!pip install -q "energy-plus-utility @ git+https://github.com/mugalan/energy-plus-utility.git@dev"

print("Dependencies installed.")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Dependencies installed.


In [ ]:
# Ensure the directory and log file exist, then tail
!mkdir -p /tmp/smarthvac_sim_runs && touch /tmp/smarthvac_sim_runs/backend.log



In [ ]:
!tail -n 100 -f /tmp/smarthvac_sim_runs/backend.log
